# Генерация текстового описания

Нужно получить связное текстовое описание видео с авторегистратора.

С помощью алгоритма распознавания объектов на изображении были получены структурные данные видеосцены.

Формат данных следующий:

```
{
    0: [
        {
            id: 0,
            class: 'main',
            bbox: [1, 2, 3, 4]
        }
    ]
}
```

Ключ-число на верхнем уровне соответствует номеру кадра

В каждом кадре содержится массив обнаруженных объектов со следующими полями:

- id - уникальный идентификатор объекта для различения между кадрами
- class - класс объекта (знак, автомобиль, пешеход и т.д.)
- bbox - координаты рамки, содержащей объект. Используется для определения размера

По этим данным нужно составить часть промпта для языковой модели.

Через ";" будут перечислены дорожные знаки, в конце количество встретившихся машин и пешеходов.

Чтобы избежать дублирования пешеходных переходов из-за двух знаков по обеим сторонам дороги, один будет отбрасываться при примерном совпадении размеров на кадре.

Пример: `переход; главная дорога; всего машин 2 пешеходов 0, длителность 5 сек.`

## Подготовка данных

In [1]:
import json
from utils import get_area, translator, is_car

with open("./data/data1.json", "r", encoding="utf-8") as f:
    data = json.load(f)

duration = 41
result = ''

car_count = 0
pedestrian_count = 0

checked_ids = {}

for items in data.values():
    crosswalks = [v for v in items if v.get("class") == "Crossroad"]
    base_items = [v for v in items if v.get("class") != "Crossroad"]

    if len(crosswalks) > 0 and not checked_ids.get(crosswalks[0].get("id")):
        checked_ids[crosswalks[0].get("id")] = True
        result += 'пешеходный переход; '

        base_area = get_area(crosswalks[0].get('bbox'))

        for i in range(1, len(crosswalks)):
            area = get_area(crosswalks[1].get('bbox'))
            if (0.8 * base_area) <= area <= (1.2 * base_area):
                checked_ids[crosswalks[i].get("id")] = True

    for item in base_items:
        if checked_ids.get(item.get("id")):
            continue

        checked_ids[item.get("id")] = True

        if is_car(item):
            car_count += 1
        elif item.get('class') == 'pedestrian':
            pedestrian_count += 1
        else:
            name = translator.get(item.get('class'))
            result += '' if not name else name + '; '

result += f'всего машин {car_count} пешеходов {pedestrian_count}, длительность {duration} сек.'

print(result)


пешеходный переход; главная дорога; главная дорога; пешеходный переход; главная дорога; главная дорога; пешеходный переход; всего машин 12 пешеходов 0, длительность 41 сек.


## Применение языковой модели

Используется модель Qwen, т.к. она сочетает в себе небольшой вес и хорошую поддержку русского языка.

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

base_prompt = f"""
Нужно связно описать видео с регистратора по объектам, попавшим на видео.
Описывай видео только по полученной информации, нужно связно перечислить объекты в виде текста.
Обязательно укажи дорожные знаки. Не включай в описание автомобили и пешеходов.
На основе количества автомобилей и пешеходов в конце сделай вывод о загруженности трафика.
"""

class QwenDescriber:
    def __init__(self, size="1.5b"):
        self.model_name = f"Qwen/Qwen2.5-{size}-Instruct"
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            dtype=torch.float16,
            device_map="auto"
        )

    def generate_description(self, prompt):
        messages = [
            {"role": "system", "content": base_prompt},
            {"role": "user", "content": prompt}
        ]

        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)

        outputs = self.model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7
        )

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

# describer = QwenDescriber()
#
# print(describer.generate_description(result))

## Заключение

В целом полученное текстовое описание неплохое. Перечислены объекты и сделан вывод, текст связный. Но нужно отметить, что нейросеть выдала и ложную информацию о 12 пешеходах, хотя их было 0. Нужно учитывать, что это самая простая модель Qwen - 2.5b токенов, поэтому результаты крайне не точные. Были замечены артефакты в виде китайских и английских символов.

Для сравнения приведен ответ текущей версии LLM Deepseek:
```
На видео зафиксирован участок дороги с чередующимися зонами. Основным элементом является главная дорога, которая последовательно проходит через всю запись. Дорога
несколько раз пересекается с пешеходными переходами, обозначенными соответствующей разметкой «зебра». В кадре присутствуют дорожные знаки: «Главная дорога»
и «Пешеходный переход».

За 41 секунду записи в кадре было отмечено 12 автомобилей при полном отсутствии пешеходов. Вывод: трафик на данном участке умеренно загруженный, движение транспорта
непрерывное, однако пешеходная активность в данный момент времени отсутствует.
```

Заметно, что текст стал гораздо лучше, также нет ложной информации. Можно сделать вывод, что с помощью увелечения мощностей машины и соответственно более продвинутой LLM результаты станут полезными и сопоставимыми с человеческим описанием.

In [4]:
from transformers import TextIteratorStreamer
from threading import Thread

class StreamingQwenDescriber(QwenDescriber):
    def generate_stream(self, prompt, max_tokens=200):
        streamer = TextIteratorStreamer(
            self.tokenizer,
            skip_prompt=True,
            skip_special_tokens=True
        )

        messages = [
            {"role": "system", "content": base_prompt},
            {"role": "user", "content": prompt}
        ]

        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)

        generation_kwargs = dict(
            **inputs,
            streamer=streamer,
            max_new_tokens=max_tokens,
            temperature=0.7,
            do_sample=True
        )

        thread = Thread(target=self.model.generate, kwargs=generation_kwargs)
        thread.start()

        return streamer

describer = StreamingQwenDescriber("1.5b")
streamer = describer.generate_stream(result)

for token in streamer:
    print(token, end="", flush=True)

Some parameters are on the meta device because they were offloaded to the cpu and disk.


В этом видео можно заметить следующие объекты:
- Пешеходный переход;
- Главная дорога.

Долгий просмотр показал, что всего было зарегистрировано 12 машин и 0 пешеходов за время 41 секунду. В результате, можно сделать вывод о загруженности трафика как следующее: данное место является довольно загруженным, так как из 12 движущихся объектов все они являются автомобилями, а пешеходов нет.